#### SG2 swerve base keyboard teleop — `sim`

베이스만 키보드로 움직여 보는 시뮬레이션. 같은 입력/속도 제한으로 `real_base_controller_sg2.ipynb` 에서 실제 로봇을 움직입니다.

| 키 (MuJoCo 창 클릭 후) | 동작 |
|---|---|
| `W` / `S` | 전진 / 후진 |
| `A` / `D` | 왼쪽 / 오른쪽 (옆으로) |
| `Q` / `E` | 좌회전 / 우회전 |
| `SPACE` | 즉시 정지 |

- 최대 속도 0.3 m/s, 0.6 rad/s (inbound.py 와 동일) × `Speed` (SLOW 0.3 / NORMAL 0.6 / FAST 1.0)
- 가속 제한 0.5 m/s², 1.0 rad/s² — 키를 떼면 서서히 멈춤
- 바퀴 조향/회전은 실제 `swerve_drive_controller` 와 같은 모듈 위치로 계산

In [9]:
import os
if "__vsc_ipynb_file__" in globals(): os.chdir(os.path.dirname(__vsc_ipynb_file__))   # VS Code 커널 cwd 보정
%run ../package/init_project.py

import numpy as np

from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *
from utils import *
from package.base_control import *   # not `from base_control`: PACKAGE_DIR is on sys.path only during %run

app = get_qt_app()


[ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation


In [10]:
xml_path = merge_mjcfs(
    included_mjcf_files = [
        str(FLOOR_XML_PATH),
        str(SG2_XML_PATH),
    ],
    output_xml_path = str(SG2_SCENE_XML_PATH),)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=False)
env.reset()
base_z = get_sg2_base_z(env)
print("steer joints:",SG2_BASE_STEER_JOINT_NAMES)
print("drive joints:",SG2_BASE_DRIVE_JOINT_NAMES)

[merge_mjcfs] Merging [2] MJCF files:
  - [0] [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/floor_simple_white.xml]
  - [1] [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/ffw_sg2.xml]
[merge_mjcfs] Saved merged XML to:[/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml]
MuJoCo model loaded from [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].
steer joints: ['left_wheel_steer_joint', 'right_wheel_steer_joint', 'rear_wheel_steer_joint']
drive joints: ['left_wheel_drive_joint', 'right_wheel_drive_joint', 'rear_wheel_drive_joint']


In [11]:
env.init_viewer(title="SG2 Base Teleop (sim)",x_offset=0.23,y_offset=0.00,width=0.77,height=1.0,fontscale=180)
env.viewer.set_cam_info(200.0,4.0,-35.0,[0.0,0.0,0.8])

teleop = SG2BaseTeleop()
ctrl_radio = create_sg2_base_command_widget(real=False)

In [12]:
hz = 30
tmr = SimpleTimer(name="Base Sim",Hz=hz,verbose=False)
tmr.start(); env.reset_wall_time(); env.reset_sim_time()

try:
    while env.is_viewer_alive() and ctrl_radio.isVisible():
        app.processEvents()

        if tmr.do_run():
            teleop.scale = SG2_BASE_SPEED_SCALES[ctrl_radio.get_button_option(label_text="Speed")]
            teleop.read_keys(env)
            teleop.update(1.0/hz)
            apply_sg2_base_to_env(env,teleop,base_z)

            if ctrl_radio.get_button_option(label_text="Camera") == "FOLLOW":
                env.viewer.set_cam_info(lookat=[teleop.pose[0],teleop.pose[1],0.8])
            env.plot_global_coordinate_axes()
            draw_sg2_base(env,teleop,base_z)
            env.viewer_text_overlay("Base",make_sg2_base_overlay_text(teleop),loc="top left")
            env.render()
            tmr.end()
except KeyboardInterrupt:
    print("Interrupted.")
finally:
    ctrl_radio.close()
    env.close_viewer()
    print("final pose x %.2f y %.2f yaw %.1fdeg"%(teleop.pose[0],teleop.pose[1],np.degrees(teleop.pose[2])))
    print("Done.")

Interrupted.
final pose x 0.00 y 0.00 yaw 0.0deg
Done.
